# 💰 Personal Expense Analytics (2024)
> **Internship Project | Data Analytics Domain | CodTech**  
> Author: [Your Name] | Tools: Python · Pandas · NumPy · Matplotlib · Seaborn

---
## Table of Contents
1. [Setup & Imports](#1)
2. [Generate / Load Dataset](#2)
3. [Data Cleaning](#3)
4. [Exploratory Data Analysis](#4)
5. [Visualizations](#5)
6. [Savings Analysis](#6)
7. [Key Insights & Conclusions](#7)

---
## 1. Setup & Imports <a id='1'></a>

In [ ]:
import os, sys, random, warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from datetime import datetime, timedelta

# Add src to path so we can import analysis.py helpers
sys.path.insert(0, os.path.join('..', 'src'))

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams.update({'figure.dpi': 120, 'axes.titlesize': 13,
                     'axes.titleweight': 'bold', 'axes.labelsize': 11})

CHARTS_DIR = os.path.join('..', 'outputs', 'charts')
os.makedirs(CHARTS_DIR, exist_ok=True)
print('✅ All libraries loaded successfully!')

---
## 2. Generate / Load Dataset <a id='2'></a>
We generate **1 000 synthetic expense records** for the year 2024 covering 10 real-world spending categories.

In [ ]:
# ── Parameters ───────────────────────────────────────────────
SEED           = 42
N_RECORDS      = 1000
MONTHLY_INCOME = 45000
random.seed(SEED); np.random.seed(SEED)

categories = {
    'Food & Dining'       : (150,  80),
    'Transportation'      : (80,   40),
    'Shopping'            : (200, 120),
    'Entertainment'       : (100,  60),
    'Healthcare'          : (120,  90),
    'Education'           : (300, 150),
    'Utilities'           : (60,   20),
    'Rent'                : (8000, 500),
    'Personal Care'       : (50,   25),
    'Savings & Investment': (500, 200),
}
payment_methods = ['Cash', 'Credit Card', 'Debit Card', 'UPI', 'Net Banking']

records = []
start_date = datetime(2024, 1, 1)
date_range = (datetime(2024, 12, 31) - start_date).days
expense_id = 1

# Monthly rent
for month in range(1, 13):
    records.append({
        'ExpenseID'    : expense_id,
        'Date'         : datetime(2024, month, 1).strftime('%Y-%m-%d'),
        'Category'     : 'Rent',
        'Description'  : 'Monthly Rent',
        'Amount'       : round(np.random.normal(8000, 200), 2),
        'PaymentMethod': 'Net Banking',
        'MonthlyIncome': MONTHLY_INCOME,
    })
    expense_id += 1

# Remaining records
for _ in range(N_RECORDS - 12):
    cat    = random.choice([c for c in categories if c != 'Rent'])
    mean, std = categories[cat]
    amount = round(abs(np.random.normal(mean * 0.15, std * 0.10)) + 10, 2)
    exp_date = start_date + timedelta(days=random.randint(0, date_range))
    records.append({
        'ExpenseID'    : expense_id,
        'Date'         : exp_date.strftime('%Y-%m-%d'),
        'Category'     : cat,
        'Description'  : f'{cat} expense',
        'Amount'       : amount,
        'PaymentMethod': random.choice(payment_methods),
        'MonthlyIncome': MONTHLY_INCOME,
    })
    expense_id += 1

df_raw = pd.DataFrame(records)
df_raw['Date'] = pd.to_datetime(df_raw['Date'])
df_raw = df_raw.sort_values('Date').reset_index(drop=True)

# Save CSV
csv_path = os.path.join('..', 'data', 'expenses.csv')
df_raw.to_csv(csv_path, index=False)
print(f'✅ Dataset saved to {csv_path}')
print(f'   Shape : {df_raw.shape}')
df_raw.head(10)

---
## 3. Data Cleaning <a id='3'></a>

In [ ]:
df = df_raw.copy()

print('── Before Cleaning ──')
print(f'Shape         : {df.shape}')
print(f'Null values   :\n{df.isnull().sum()}')
print(f'Duplicates    : {df.duplicated().sum()}')

In [ ]:
# Fix dtypes
df['Date']   = pd.to_datetime(df['Date'])
df['Amount'] = pd.to_numeric(df['Amount'], errors='coerce')

# Remove invalids
df = df.dropna(subset=['Amount','Category'])
df = df[df['Amount'] > 0]
df = df.drop_duplicates(subset=['ExpenseID'])

# Derived columns
df['Month']     = df['Date'].dt.month
df['MonthName'] = df['Date'].dt.strftime('%b')
df['Year']      = df['Date'].dt.year
df['DayOfWeek'] = df['Date'].dt.day_name()
df['Week']      = df['Date'].dt.isocalendar().week.astype(int)

monthly_spend = df.groupby('Month')['Amount'].sum().rename('MonthlySpend')
df = df.merge(monthly_spend.reset_index(), on='Month', how='left')
df['MonthlySavings'] = df['MonthlyIncome'] - df['MonthlySpend']

print('── After Cleaning ──')
print(f'Shape : {df.shape}')
df.dtypes

---
## 4. Exploratory Data Analysis <a id='4'></a>

In [ ]:
print('── Descriptive Statistics ──')
df[['Amount','MonthlyIncome','MonthlySavings']].describe().round(2)

In [ ]:
print('── Category-wise Total Spend ──')
cat_spend = df.groupby('Category')['Amount'].agg(['sum','mean','count']).sort_values('sum', ascending=False)
cat_spend.columns = ['Total (₹)', 'Avg per txn (₹)', 'Transactions']
cat_spend.round(2)

In [ ]:
print('── Monthly Summary ──')
monthly = (df.groupby(['Month','MonthName'])
             .agg(Total_Spend=('Amount','sum'),
                  Transactions=('ExpenseID','count'),
                  Income=('MonthlyIncome','first'))
             .reset_index().sort_values('Month'))
monthly['Savings']     = monthly['Income'] - monthly['Total_Spend']
monthly['Savings_Pct'] = (monthly['Savings'] / monthly['Income'] * 100).round(1)
monthly[['MonthName','Total_Spend','Savings','Savings_Pct','Transactions']]

---
## 5. Visualizations <a id='5'></a>

In [ ]:
# ── Chart 1 : Monthly Spending Trend ─────────────────────────
fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(monthly['MonthName'], monthly['Total_Spend'],
        marker='o', lw=2.5, color='#2196F3', ms=8)
ax.fill_between(range(len(monthly)), monthly['Total_Spend'],
                alpha=0.12, color='#2196F3')
ax.set_xticklabels(monthly['MonthName'])
ax.set_title('Monthly Spending Trend (2024)')
ax.set_ylabel('Total Spent (₹)')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'₹{x:,.0f}'))
for i, row in monthly.iterrows():
    ax.annotate(f"₹{row['Total_Spend']:,.0f}",
                xy=(i - monthly.index[0], row['Total_Spend']),
                xytext=(0,10), textcoords='offset points',
                ha='center', fontsize=8)
fig.tight_layout()
fig.savefig(os.path.join(CHARTS_DIR,'01_monthly_trend.png'), bbox_inches='tight')
plt.show()

In [ ]:
# ── Chart 2 : Category-wise Horizontal Bar ───────────────────
cat_totals = df.groupby('Category')['Amount'].sum().sort_values()
colors = sns.color_palette('Blues_d', len(cat_totals))

fig, ax = plt.subplots(figsize=(10, 6))
bars = ax.barh(cat_totals.index, cat_totals.values, color=colors, edgecolor='white')
ax.bar_label(bars, fmt='₹%.0f', padding=5, fontsize=9)
ax.set_title('Category-wise Total Spending')
ax.set_xlabel('Total Amount (₹)')
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'₹{x:,.0f}'))
fig.tight_layout()
fig.savefig(os.path.join(CHARTS_DIR,'02_category_bar.png'), bbox_inches='tight')
plt.show()

In [ ]:
# ── Chart 3 : Category Pie Chart ─────────────────────────────
cat_spend2 = df.groupby('Category')['Amount'].sum().sort_values(ascending=False)
top5   = cat_spend2.head(5)
others = cat_spend2.iloc[5:].sum()
labels = list(top5.index) + ['Others']
sizes  = list(top5.values) + [others]

fig, ax = plt.subplots(figsize=(8, 8))
ax.pie(sizes, labels=labels, autopct='%1.1f%%', startangle=140,
       wedgeprops={'edgecolor':'white','linewidth':2},
       colors=sns.color_palette('pastel', len(labels)))
ax.set_title('Expense Distribution — Top 5 Categories + Others')
fig.tight_layout()
fig.savefig(os.path.join(CHARTS_DIR,'03_category_pie.png'), bbox_inches='tight')
plt.show()

In [ ]:
# ── Chart 4 : Savings vs Spending Grouped Bar ────────────────
x     = np.arange(len(monthly))
width = 0.35

fig, ax = plt.subplots(figsize=(12, 6))
ax.bar(x - width/2, monthly['Total_Spend'], width,
       label='Spending', color='#EF5350', alpha=0.85)
ax.bar(x + width/2, monthly['Savings'], width,
       label='Savings',  color='#66BB6A', alpha=0.85)
ax.set_xticks(x)
ax.set_xticklabels(monthly['MonthName'])
ax.set_title('Monthly Spending vs Savings')
ax.set_ylabel('Amount (₹)')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'₹{x:,.0f}'))
ax.axhline(MONTHLY_INCOME, linestyle='--', color='steelblue',
           linewidth=1.5, label=f'Income ₹{MONTHLY_INCOME:,}')
ax.legend()
fig.tight_layout()
fig.savefig(os.path.join(CHARTS_DIR,'04_savings_vs_spending.png'), bbox_inches='tight')
plt.show()

In [ ]:
# ── Chart 5 : Payment Method Donut ───────────────────────────
pm = df.groupby('PaymentMethod')['Amount'].sum().sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(7, 7))
ax.pie(pm.values, labels=pm.index, autopct='%1.1f%%', startangle=90,
       wedgeprops={'edgecolor':'white','linewidth':2.5,'width':0.5},
       colors=sns.color_palette('Set2', len(pm)))
ax.set_title('Spending by Payment Method')
fig.tight_layout()
fig.savefig(os.path.join(CHARTS_DIR,'05_payment_method_donut.png'), bbox_inches='tight')
plt.show()

In [ ]:
# ── Chart 6 : Weekly Heatmap ─────────────────────────────────
dow_order = ['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday']
pivot = (df.groupby(['DayOfWeek','Week'])['Amount']
           .sum().unstack(fill_value=0))
pivot = pivot.reindex(dow_order)

fig, ax = plt.subplots(figsize=(18, 5))
sns.heatmap(pivot, ax=ax, cmap='YlOrRd', linewidths=0.3,
            linecolor='white', cbar_kws={'label':'₹ Spent'})
ax.set_title('Weekly Spending Heatmap (Day-of-Week × Week Number)')
fig.tight_layout()
fig.savefig(os.path.join(CHARTS_DIR,'06_weekly_heatmap.png'), bbox_inches='tight')
plt.show()

In [ ]:
# ── Chart 7 : Top 5 Stacked Monthly Bar ──────────────────────
top5_cats = df.groupby('Category')['Amount'].sum().nlargest(5).index.tolist()
sub   = df[df['Category'].isin(top5_cats)]
pivot2 = sub.groupby(['Month','Category'])['Amount'].sum().unstack(fill_value=0)
pivot2.index = [datetime(2024, m, 1).strftime('%b') for m in pivot2.index]

fig, ax = plt.subplots(figsize=(12, 6))
pivot2.plot(kind='bar', stacked=True, ax=ax,
            colormap='tab10', edgecolor='white', linewidth=0.5)
ax.set_title('Top 5 Categories — Monthly Stacked Spending')
ax.set_xlabel('Month'); ax.set_ylabel('Amount (₹)')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'₹{x:,.0f}'))
ax.legend(loc='upper right', fontsize=9)
plt.xticks(rotation=45)
fig.tight_layout()
fig.savefig(os.path.join(CHARTS_DIR,'07_top5_stacked_bar.png'), bbox_inches='tight')
plt.show()

---
## 6. Savings Analysis <a id='6'></a>

In [ ]:
total_income = MONTHLY_INCOME * 12
total_spent  = df['Amount'].sum()
total_saved  = total_income - total_spent
savings_rate = (total_saved / total_income) * 100

print('──── Annual Savings Summary ────')
print(f'  Annual Income  : ₹{total_income:>12,.2f}')
print(f'  Total Spent    : ₹{total_spent:>12,.2f}')
print(f'  Total Saved    : ₹{total_saved:>12,.2f}')
print(f'  Savings Rate   : {savings_rate:.1f}%')

if savings_rate < 10:
    print('  ⚠ ALERT: Savings rate critically low (<10%)')
elif savings_rate < 20:
    print('  ⚠ Savings rate below recommended 20% threshold')
else:
    print('  ✅ Healthy savings rate!')

In [ ]:
print('\n──── Monthly Savings Rate ────')
monthly[['MonthName','Total_Spend','Savings','Savings_Pct']].to_string(index=False)

---
## 7. Key Insights & Conclusions <a id='7'></a>

### 📊 Summary of Findings

| Metric | Value |
|--------|-------|
| Total Annual Income | ₹5,40,000 |
| Total Annual Expenditure | Computed above |
| Savings Rate | Computed above |
| Top Expense Category | Rent |
| Most Used Payment Method | Varies |

### 🔍 Key Observations

1. **Rent dominates** total spending, which is typical for urban salaried individuals.
2. **Shopping and Education** are the second and third highest categories, suggesting opportunities to reduce discretionary spending.
3. **Weekend spending** is noticeably higher than weekday spending (visible in heatmap).
4. **UPI and Credit Card** are the most commonly used payment methods.
5. **Savings rate varies monthly** — some months show negative savings when one-time expenses spike.

### ✅ Recommendations

- **Follow the 50/30/20 rule**: 50% needs, 30% wants, 20% savings.
- **Automate savings**: Set up auto-debit SIP on salary day.
- **Reduce Entertainment + Shopping** spends by 15–20% to boost savings.
- **Use a budget tracker** app to stay under category limits.
- **Build emergency fund** = 6× monthly expenses before investing.

---
*Project completed as part of Data Analytics Internship at CodTech.*